In [20]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing import sequence 
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN, Input
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import imdb
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import tensorflow as tf
import numpy as np
from scipy.stats import randint, loguniform
 

In [2]:
sent = [ ' the glass of milk',
        'the glass of juice',
        'the cup of tea',
        'I am a good boy',
        'understand the meaning of words',
        'your videos are good'
]

In [3]:
sent

[' the glass of milk',
 'the glass of juice',
 'the cup of tea',
 'I am a good boy',
 'understand the meaning of words',
 'your videos are good']

In [4]:
voc_size = 10000

In [5]:
one_hot_repr = [one_hot(words, voc_size) for words in sent]
one_hot_repr


[[7076, 6079, 5407, 447],
 [7076, 6079, 5407, 72],
 [7076, 6398, 5407, 463],
 [1438, 8192, 7654, 7086, 4225],
 [8285, 7076, 5533, 5407, 1307],
 [4354, 4193, 4162, 7086]]

In [6]:
sent_length = 8
embedded_docs = pad_sequences(one_hot_repr, padding = 'pre', maxlen = sent_length)
print(embedded_docs)

[[   0    0    0    0 7076 6079 5407  447]
 [   0    0    0    0 7076 6079 5407   72]
 [   0    0    0    0 7076 6398 5407  463]
 [   0    0    0 1438 8192 7654 7086 4225]
 [   0    0    0 8285 7076 5533 5407 1307]
 [   0    0    0    0 4354 4193 4162 7086]]


In [7]:
dim = 10
model = Sequential()
model.add(Embedding(voc_size, dim, input_shape = (sent_length, )))
model.compile('adam', 'mse')
model.summary()

c:\Users\tariq\Desktop\NLP MLO\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 8, 10)          │       100,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,000 (390.62 KB)

 Trainable params: 100,000 (390.62 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.predict(embedded_docs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step


array([[[ 0.01718592, -0.02178468, -0.04152427, -0.01716415,
         -0.02189994, -0.04592905, -0.043344  ,  0.03886541,
          0.03227406,  0.01193001],
        [ 0.01718592, -0.02178468, -0.04152427, -0.01716415,
         -0.02189994, -0.04592905, -0.043344  ,  0.03886541,
          0.03227406,  0.01193001],
        [ 0.01718592, -0.02178468, -0.04152427, -0.01716415,
         -0.02189994, -0.04592905, -0.043344  ,  0.03886541,
          0.03227406,  0.01193001],
        [ 0.01718592, -0.02178468, -0.04152427, -0.01716415,
         -0.02189994, -0.04592905, -0.043344  ,  0.03886541,
          0.03227406,  0.01193001],
        [ 0.00720615,  0.0247658 ,  0.02370426, -0.001931  ,
          0.01197101,  0.02138707,  0.00709341, -0.00661173,
          0.00771894, -0.03838889],
        [ 0.03040821, -0.01288506,  0.03461324,  0.02496301,
          0.03701297, -0.03008759, -0.02998613, -0.01822606,
         -0.02330331, -0.04453386],
        [-0.0208599 , -0.02590438, -0.03820769,  0.0

In [9]:
max_features = 10000
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words = max_features)

print(f'Training data shape: {x_train.shape}, Training label shape: {y_train.shape}')
print(f'Training data shape: {x_test.shape}, Training label shape: {y_test.shape}')

c:\Users\tariq\Desktop\NLP MLO\venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Training data shape: (25000,), Training label shape: (25000,)
Training data shape: (25000,), Training label shape: (25000,)


In [10]:
sample_review = x_train[0]
sample_label = y_train[0]

print(f'Sample review (as integers): {sample_review}')
print(f'Sample label: {sample_label}')

Sample review (as integers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Sample label: 1


In [11]:
word_index = imdb.get_word_index()
word_index
reverse_word_index = {value: key for key, value in word_index.items()}
reverse_word_index

{34701: 'fawn',
 52006: 'tsukino',
 52007: 'nunnery',
 16816: 'sonja',
 63951: 'vani',
 1408: 'woods',
 16115: 'spiders',
 2345: 'hanging',
 2289: 'woody',
 52008: 'trawling',
 52009: "hold's",
 11307: 'comically',
 40830: 'localized',
 30568: 'disobeying',
 52010: "'royale",
 40831: "harpo's",
 52011: 'canet',
 19313: 'aileen',
 52012: 'acurately',
 52013: "diplomat's",
 25242: 'rickman',
 6746: 'arranged',
 52014: 'rumbustious',
 52015: 'familiarness',
 52016: "spider'",
 68804: 'hahahah',
 52017: "wood'",
 40833: 'transvestism',
 34702: "hangin'",
 2338: 'bringing',
 40834: 'seamier',
 34703: 'wooded',
 52018: 'bravora',
 16817: 'grueling',
 1636: 'wooden',
 16818: 'wednesday',
 52019: "'prix",
 34704: 'altagracia',
 52020: 'circuitry',
 11585: 'crotch',
 57766: 'busybody',
 52021: "tart'n'tangy",
 14129: 'burgade',
 52023: 'thrace',
 11038: "tom's",
 52025: 'snuggles',
 29114: 'francesco',
 52027: 'complainers',
 52125: 'templarios',
 40835: '272',
 52028: '273',
 52130: 'zaniacs',

In [12]:
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_review])
decoded_review

"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

In [13]:
max_len = 500 

x_train = pad_sequences(x_train, maxlen = max_len)
x_test = pad_sequences(x_test, maxlen = max_len)

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate = 1e-3)
loss = tf.keras.losses.BinaryCrossentropy()
earlystopping = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

def create_model(max_features:int, max_len:int, embed_dim:int = 128, rnn_units:int = 128, rnn_layers:int = 1, dropout:float = 0.0, recurrent_dropout:float = 0.0, learning_rate:float = opt):
    model = Sequential()
    model.add(Embedding(input_dim = max_features, output_dim = embed_dim, input_shape = (max_len,)))

    for i in range(rnn_layers):
        return_seq = (i < rnn_layers - 1)
        model.add(SimpleRNN(units = rnn_units,
                             activation = 'relu',
                             return_sequences = return_seq,
                             dropout = dropout, 
                             recurrent_dropout = recurrent_dropout))

    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer = opt, loss = loss, metrics = ['accuracy'])

    return model

model = KerasClassifier(model = create_model, max_features = max_features, max_len = max_len, verbose = 0, callbacks=[earlystopping], validation_split = 0.2)

param_grid = {
    'model__embed_dim': [128, 256],
    'model__rnn_units': randint(64, 256),
    'model__rnn_layers': [1, 2],
    'model__learning_rate': loguniform(1e-4, 1e-2),
    'batch_size': [32, 64, 128],
    'epochs': [20]
}

grid = RandomizedSearchCV(estimator = model, param_distributions = param_grid, n_jobs = -1, n_iter = 10, cv = 3)
grid_result = grid.fit(x_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

c:\Users\tariq\Desktop\NLP MLO\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Best: 0.793400 using {'batch_size': 64, 'epochs': 20, 'model__embed_dim': 128, 'model__learning_rate': np.float64(0.0031504581624325063), 'model__rnn_layers': 2, 'model__rnn_units': 124}


In [46]:
model = Sequential()
model.add(Input(shape = (max_len, )))
model.add(Embedding(max_features, 128))
model.add(SimpleRNN(124, activation = 'tanh', return_sequences = True))
model.add(SimpleRNN(124, activation = 'tanh'))
model.add(Dense(1, activation = 'sigmoid'))
model.summary()



Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_18 (SimpleRNN)       │ (None, 500, 124)       │        31,372 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_19 (SimpleRNN)       │ (None, 124)            │        30,876 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,342,373 (5.12 MB)

 Trainable params: 1,342,373 (5.12 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
opt = tf.keras.optimizers.Adam(learning_rate = 0.0031504581624325063) #learning_rate = 0.0031504581624325063
loss = tf.keras.losses.BinaryCrossentropy()

model.compile(optimizer = opt, loss = loss, metrics = ['accuracy'])
earlystopping = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

In [48]:
history = model.fit(
    x_train, y_train, epochs = 10, batch_size = 32,
    validation_split = 0.2,
    callbacks = [earlystopping],
    verbose = 1
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 129ms/step - accuracy: 0.5046 - loss: 0.7090 - val_accuracy: 0.5046 - val_loss: 0.6971
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 79s 127ms/step - accuracy: 0.5030 - loss: 0.7004 - val_accuracy: 0.5068 - val_loss: 0.6959
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 79s 126ms/step - accuracy: 0.5842 - loss: 0.6702 - val_accuracy: 0.5422 - val_loss: 0.6780
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 86s 138ms/step - accuracy: 0.6598 - loss: 0.6118 - val_accuracy: 0.5700 - val_loss: 0.7016
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 95s 152ms/step - accuracy: 0.6963 - loss: 0.5802 - val_accuracy: 0.6518 - val_loss: 0.6329
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 104s 166ms/step - accuracy: 0.7638 - loss: 0.5036 - val_accuracy: 0.6738 - val_loss: 0.6304
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 91s 146ms/step - accuracy: 0.7842 - loss: 0.4712 - val_accuracy: 0.6712 - val_loss: 0.6291
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 88s 141ms/step - accuracy: 0.7922 - loss: 

In [50]:
model.save('simple_rnn_imdb.h5')